# 🌐 AI 공급망 미니-GEM: ECM 장·단기 탄력성 Playground

Oxford Economics **GEM의 국가 방정식이 쓰는 오차수정모형(ECM)** 을 AI 공급망 밸류체인에 적용합니다. 각 고객→공급사 관계에서 공급사 매출이 고객 활동에 어떻게 **장기 균형 + 단기 조정**하는지 추정하고, **충격 시나리오**를 시뮬레이션합니다.

**모형 (Engle-Granger 2단계):**
1. 장기: `log(공급사 매출) = c + θ·log(고객 활동) + 계절더미`  → **θ = 장기 탄력성**
2. 공적분: 잔차 ADF 검정 (numpy)
3. 단기: `Δlog(Y) = α + β·Δlog(X) + λ·EC_{t-1}`  → **λ = 조정속도**(<0·유의 = 오차수정), **반감기**

**GEM 대응:** 국가↔섹션 · 무역흐름↔밸류체인 관계 · ECM↔ECM · 시나리오↔충격 IRF

- **엔진:** `ecm.py` (numpy) · **데이터:** `panel_long` · **관계:** `valuechain.py`
- ⚠️ **한계:** 표본 ≈40분기 + AI붐 구조변화 → 정식 공적분 검출력은 낮음. θ 유의성 + λ 부호로 신뢰도 판단.

## Step 1 — 데이터 로드

In [ ]:
%matplotlib inline
import pandas as pd
from IPython.display import display
import panel as P, ecm as E
pd.set_option('display.max_rows', 60, 'display.width', 200)
long = P.load_long(P.DB_DEFAULT)
print('panel_long:', long.shape)

## Step 2 — 한 관계 ECM 상세 (핵심)

고객·공급사 섹션과 드라이버 변수(X)를 지정하면 장기 θ, 조정 λ, 반감기, 공적분 검정, 시나리오 IRF를 모두 계산합니다.

In [ ]:
# ============== EDIT ==============
CUSTOMER = 'hyperscalers'   # 고객 섹션 (수요/투자 주체)
SUPPLIER = 'dram'           # 공급사 섹션 (매출 Y)
X_ITEM   = 'capex'          # 고객 드라이버: revenue / capex / cogs
SHOCK    = 10               # 영구 충격 %
# =================================

r = E.estimate_edge(long, CUSTOMER, SUPPLIER, X_ITEM, 'revenue')
display(pd.Series({k:v for k,v in r.items() if not k.startswith('_')}))
E.plot_ecm(long, CUSTOMER, SUPPLIER, X_ITEM, 'revenue', shock_pct=SHOCK);

## Step 3 — 전체 밸류체인 ECM (드라이버 비교)

모든 고객→공급사 관계에 대해 장·단기 탄력성을 추정합니다. `error_correcting=True`(λ<0·유의)인 관계가 GEM식 전이가 확인된 곳입니다.

In [ ]:
DRIVER = 'revenue'          # 'revenue' 또는 'capex'
df = E.run_ecm(long, x_item=DRIVER, y_item='revenue')
ok = df[df.status=='ok']
print(f'분석가능 {len(ok)}/{len(df)} · 오차수정 {int((ok.error_correcting==True).sum())}건')
display(ok.sort_values('theta_t', ascending=False)[
  ['customer','supplier','long_run_elasticity','theta_t','adjustment_lambda','lambda_t','half_life_q','error_correcting']])

## Step 4 — 충격 시나리오 (여러 관계 비교)

고객 활동 영구 +10% 충격이 각 공급사 매출로 전이되는 경로를 비교합니다 (GEM의 시나리오 시뮬레이션에 대응).

In [ ]:
import matplotlib.pyplot as plt
edges = [('hyperscalers','dram','capex'), ('hyperscalers','ai_chip','capex'),
         ('ai_chip','osat_packaging','revenue'), ('hyperscalers','server_networking','revenue')]
fig, ax = plt.subplots(figsize=(9,5))
for c,s,xi in edges:
    r = E.estimate_edge(long, c, s, xi, 'revenue')
    if r.get('status')!='ok': continue
    irf = E.scenario_irf(r, shock_pct=10, horizon=10)
    ax.plot(irf.quarter_ahead, irf.Y_response_pct, '-o', ms=3,
            label=f"{E.V.SECTIONS[c]} {xi} → {E.V.SECTIONS[s]} (θ={r['long_run_elasticity']})")
ax.axhline(0, color='grey', lw=.6); ax.set_xlabel('분기 후'); ax.set_ylabel('공급사 매출 반응 %')
ax.set_title('고객 활동 영구 +10% 충격 → 공급사 매출 전이 경로'); ax.legend(fontsize=8); plt.show()

## Step 5 — 리포트(MD) 생성

In [ ]:
import os
os.makedirs('reports', exist_ok=True)
print('저장:', E.generate_md(long, 'reports/ecm_minigem_2026-07-06.md'))